<a href="https://www.kaggle.com/code/ab0y04/skin-lesion-imagenet?scriptVersionId=344123803" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# ===== FULL REBUILD (new session) + STAGE 16 FOLD 4: EFFICIENTNETB0 ONLY, fresh start with full runway =====
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'   # MUST precede tensorflow import
import random, gc
import numpy as np
import pandas as pd
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"Seed {SEED} set, TF {tf.__version__}, tf.keras module: {tf.keras.__name__}")
assert 'tf_keras' in tf.keras.__name__, "STOP: Keras 3 active, not legacy. Restart before continuing."

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, recall_score, confusion_matrix
print("Imports ready")

FINAL_CLASSES = ['bcc', 'bkl', 'df', 'melanoma', 'nevus', 'vasc']
NUM_CLASSES = 6
IMG_SIZE, BATCH_SIZE = 224, 32
N_FOLDS, CURRENT_FOLD = 5, 4
AUG = dict(rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
           horizontal_flip=True, zoom_range=0.1)
print(f"Config loaded, CURRENT_FOLD = {CURRENT_FOLD}")

CV_ASSIGN_PATH = '/kaggle/input/datasets/ab0y04/cvfoldassignments/cv_fold_assignments.csv'
assert os.path.exists(CV_ASSIGN_PATH), f"STOP: file not found at {CV_ASSIGN_PATH}, check the dataset is attached"
cv_assignments = pd.read_csv(CV_ASSIGN_PATH)
print(f"Loaded cv_fold_assignments.csv: {len(cv_assignments):,} rows (expect 23,836)")
assert len(cv_assignments) == 23836, "STOP: row count mismatch"

def build_fold_split(cv_assignments, fold_num, seed=42):
    test_df = cv_assignments[cv_assignments['fold'] == fold_num].reset_index(drop=True)
    remaining = cv_assignments[cv_assignments['fold'] != fold_num].reset_index(drop=True)
    remaining = remaining.copy()
    fallback = pd.Series('unlinked_' + remaining.index.astype(str), index=remaining.index)
    remaining['_split_key'] = remaining['group_id'].fillna(fallback)
    groups = remaining.groupby('_split_key')['label'].first().reset_index()
    tr_groups, va_groups = train_test_split(groups, test_size=0.15, stratify=groups['label'], random_state=seed)
    train_df = remaining[remaining['_split_key'].isin(tr_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    val_df = remaining[remaining['_split_key'].isin(va_groups['_split_key'])].drop(columns=['_split_key']).reset_index(drop=True)
    return train_df, val_df, test_df

train_df, val_df, test_df = build_fold_split(cv_assignments, CURRENT_FOLD, seed=SEED)
print(f"\n===== FOLD {CURRENT_FOLD} SPLIT =====")
print(f"Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}")
print("(expect to match custom CNN's fold 4 run: n_test 4683)")

test_groups = set(test_df['group_id'].dropna())
train_groups = set(train_df['group_id'].dropna())
val_groups = set(val_df['group_id'].dropna())
assert test_groups.isdisjoint(train_groups) and test_groups.isdisjoint(val_groups) and train_groups.isdisjoint(val_groups), \
    f"STOP: FOLD {CURRENT_FOLD} LEAKAGE detected"
print(f"Fold {CURRENT_FOLD} leakage check: PASS")

cls = np.array(FINAL_CLASSES)
cw = compute_class_weight('balanced', classes=cls, y=train_df['label'])
w_map = {c: w for c, w in zip(FINAL_CLASSES, cw)}
train_df['sample_weight'] = train_df['label'].map(w_map)
print(f"Fold {CURRENT_FOLD} class_weight:", {c: round(w,3) for c,w in zip(cls, cw)})

def make_fold_gens(preprocess_fn):
    train_idg = ImageDataGenerator(preprocessing_function=preprocess_fn, **AUG)
    eval_idg  = ImageDataGenerator(preprocessing_function=preprocess_fn)
    common = dict(x_col='image_path', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
                  batch_size=BATCH_SIZE, class_mode='categorical', classes=FINAL_CLASSES)
    tr = train_idg.flow_from_dataframe(train_df, shuffle=True, seed=SEED, weight_col='sample_weight', **common)
    va = eval_idg.flow_from_dataframe(val_df, shuffle=False, **common)
    te = eval_idg.flow_from_dataframe(test_df, shuffle=False, **common)
    return tr, va, te

def build_pretrained(base_class, num_classes=6, shape=(224,224,3)):
    base = base_class(include_top=False, weights='imagenet', input_shape=shape)
    model = Sequential([base, GlobalAveragePooling2D(), Dense(256,activation='relu'),
                          Dropout(0.3), Dense(num_classes,activation='softmax')])
    return model, base

def macro_specificity(y_true, y_pred, n_classes):
    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))
    total = cm.sum(); specs = []
    for i in range(n_classes):
        tp = cm[i,i]; fn = cm[i,:].sum()-tp; fp = cm[:,i].sum()-tp
        tn = total-tp-fn-fp
        specs.append(tn/(tn+fp) if (tn+fp)>0 else np.nan)
    return np.nanmean(specs)

print(f"\n===== Fold {CURRENT_FOLD} setup verified. Training EfficientNetB0, fresh start. =====\n")

tr, va, te = make_fold_gens(eff_pre)
print("class_indices:", te.class_indices)

model, base = build_pretrained(EfficientNetB0)
log_file = f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_log.csv'

base.trainable = False
model.compile(Adam(1e-3), 'categorical_crossentropy', ['accuracy'])
model.fit(tr, validation_data=va, epochs=10, callbacks=[CSVLogger(log_file, append=False)], verbose=1)
print("Phase 1 sanity:", model.evaluate(te, verbose=0))

base.trainable = True
model.compile(Adam(1e-5), 'categorical_crossentropy', ['accuracy'])
cbs = [EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
       ModelCheckpoint(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras', monitor='val_accuracy', save_best_only=True),
       CSVLogger(log_file, append=True)]
model.fit(tr, validation_data=va, epochs=60, callbacks=cbs, verbose=1)

y_true = np.asarray(te.classes)
y_prob = model.predict(te, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
np.savez(f'/kaggle/working/cv_f{CURRENT_FOLD}_preds_eff.npz', y_true=y_true, y_pred=y_pred, y_prob=y_prob)

reloaded = load_model(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff.keras')
verify_acc = reloaded.evaluate(te, verbose=0)[1]
live_acc = accuracy_score(y_true, y_pred)
print(f"Checkpoint verify: reloaded acc {verify_acc:.4f} vs live acc {live_acc:.4f}  match={abs(verify_acc-live_acc)<1e-3}")
del reloaded; gc.collect(); tf.keras.backend.clear_session()

result_row = dict(fold=CURRENT_FOLD, arch='eff', accuracy=live_acc,
    macro_f1=f1_score(y_true,y_pred,average='macro'),
    macro_auc=roc_auc_score(np.eye(NUM_CLASSES)[y_true], y_prob, average='macro', multi_class='ovr'),
    macro_sensitivity=recall_score(y_true,y_pred,average='macro'),
    macro_specificity=macro_specificity(y_true,y_pred,NUM_CLASSES), n_test=len(y_true))

FOLD_ACC_EFF = [0.7200496585971446, 0.7333607230895645, 0.7153975599495161]
prior_mean = np.mean(FOLD_ACC_EFF)
deviation = abs(live_acc - prior_mean) * 100
flag = "  <-- FLAG: deviates >5pp from folds 1-3 mean" if deviation > 5 else "  (within normal range)"
print(f"\nFold {CURRENT_FOLD} vs Folds 1-3 mean: {live_acc:.4f} vs {prior_mean:.4f}, deviation {deviation:.1f}pp{flag}")

pd.DataFrame([result_row]).to_csv(f'/kaggle/working/cv_f{CURRENT_FOLD}_eff_result.csv', index=False)
print(f"\nRESULT: {result_row}")
print(f"Saved: /kaggle/working/cv_f{CURRENT_FOLD}_eff_result.csv")
print(">>> DOWNLOAD THIS FILE TO YOUR COMPUTER NOW. <<<")

2026-08-22 09:20:05.953915: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787390406.174207      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787390406.237617      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1787390406.760330      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787390406.760374      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1787390406.760377      23 computation_placer.cc:177] computation placer alr

Seed 42 set, TF 2.19.0, tf.keras module: tf_keras.api._v2.keras
Imports ready
Config loaded, CURRENT_FOLD = 4
Loaded cv_fold_assignments.csv: 23,836 rows (expect 23,836)

===== FOLD 4 SPLIT =====
Train 16,262 | Val 2,891 | Test 4,683
(expect to match custom CNN's fold 4 run: n_test 4683)
Fold 4 leakage check: PASS
Fold 4 class_weight: {np.str_('bcc'): np.float64(1.209), np.str_('bkl'): np.float64(1.519), np.str_('df'): np.float64(15.943), np.str_('melanoma'): np.float64(0.872), np.str_('nevus'): np.float64(0.308), np.str_('vasc'): np.float64(15.758)}

===== Fold 4 setup verified. Training EfficientNetB0, fresh start. =====

Found 16262 validated image filenames belonging to 6 classes.
Found 2891 validated image filenames belonging to 6 classes.
Found 4683 validated image filenames belonging to 6 classes.
class_indices: {'bcc': 0, 'bkl': 1, 'df': 2, 'melanoma': 3, 'nevus': 4, 'vasc': 5}


I0000 00:00:1787390470.813464      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787390470.819569      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


16705208/16705208 [==============================] - 0s 0us/step
Epoch 1/10


E0000 00:00:1787390480.776533      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1787390482.400388      65 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1787390484.190129      66 service.cc:152] XLA service 0x7910407be740 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787390484.190192      66 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1787390484.190199      66 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1787390484.467465      66 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


509/509 [==============================] - 553s 1s/step - loss: 1.3359 - accuracy: 0.4760 - val_loss: 1.1477 - val_accuracy: 0.5527
Epoch 2/10
509/509 [==============================] - 369s 724ms/step - loss: 1.1177 - accuracy: 0.5250 - val_loss: 1.0913 - val_accuracy: 0.5704
Epoch 3/10
509/509 [==============================] - 379s 745ms/step - loss: 0.9863 - accuracy: 0.5679 - val_loss: 1.1414 - val_accuracy: 0.5451
Epoch 4/10
509/509 [==============================] - 373s 732ms/step - loss: 0.9071 - accuracy: 0.5846 - val_loss: 1.0354 - val_accuracy: 0.5918
Epoch 5/10
509/509 [==============================] - 373s 733ms/step - loss: 0.8758 - accuracy: 0.5909 - val_loss: 1.1263 - val_accuracy: 0.5600
Epoch 6/10
509/509 [==============================] - 376s 740ms/step - loss: 0.8643 - accuracy: 0.5938 - val_loss: 0.9569 - val_accuracy: 0.6209
Epoch 7/10
509/509 [==============================] - 375s 738ms/step - loss: 0.8589 - accuracy: 0.6014 - val_loss: 1.0290 - val_accuracy:

E0000 00:00:1787394507.221547      23 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape insequential/efficientnetb0/block2b_drop/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


509/509 [==============================] - 525s 941ms/step - loss: 3.7369 - accuracy: 0.4961 - val_loss: 1.6493 - val_accuracy: 0.4704
Epoch 2/60
509/509 [==============================] - 472s 926ms/step - loss: 1.6809 - accuracy: 0.4474 - val_loss: 1.6399 - val_accuracy: 0.4421
Epoch 3/60
509/509 [==============================] - 473s 930ms/step - loss: 1.3409 - accuracy: 0.4587 - val_loss: 1.5045 - val_accuracy: 0.4680
Epoch 4/60
509/509 [==============================] - 470s 923ms/step - loss: 1.1743 - accuracy: 0.4889 - val_loss: 1.3828 - val_accuracy: 0.4991
Epoch 5/60
509/509 [==============================] - 469s 922ms/step - loss: 1.0407 - accuracy: 0.5206 - val_loss: 1.3046 - val_accuracy: 0.5144
Epoch 6/60
509/509 [==============================] - 479s 941ms/step - loss: 0.9626 - accuracy: 0.5450 - val_loss: 1.1796 - val_accuracy: 0.5593
Epoch 7/60
509/509 [==============================] - 466s 916ms/step - loss: 0.8996 - accuracy: 0.5737 - val_loss: 1.1341 - val_accura